
# QML-SleepNet — GUIDE Stage 04 Bridge FULL-GRID MetricMax v1

This notebook does **not add a new methodological component**.

The previous duplicate-safe run correctly implemented the guide's Stage-03 selection and exact
`128 → 64 → 32 → 8` classical encoder. However, it used a two-step search:

1. choose the Stage-03 selector using dropout `0.4`;
2. tune dropout only for that selected selector.

That can miss a better **guide-permitted selector × dropout combination**.

This notebook therefore completes the existing search over exactly the candidates already used:

- Stage-03 selector: `S160`, `S176`, `S192`
- Dropout: `0.3`, `0.4`, `0.5`

All nine combinations are compared with the **same fold-specific random seed** so the comparison
is not confounded by different initializations.

It reuses the already-computed duplicate-safe `Z128` fold artifacts, so ANOVA/mRMR/SHAP/PCA are
**not recomputed during the 9×5 screen**.

The winning combination is selected by:

**mean Accuracy → mean AUROC → mean F1**

which matches the user's stated headline-metric priority while retaining ranking/F1 tie-breaks.

Official `x01–x35` labels are never used for model selection or training.


In [ ]:

# Cell 1 — environment and paths
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import gc, json, math, os, random, hashlib, warnings
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, average_precision_score,
    matthews_corrcoef
)

import xgboost as xgb
from xgboost import XGBClassifier

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ROOT = Path("/content/drive/MyDrive/QML_SleepNet")
S3 = ROOT / "outputs/GUIDE_EXACT_METRICMAX/03_feature_bank_v1"
PREV = ROOT / "outputs/GUIDE_EXACT_METRICMAX/04_bridge128to8_v1_2_duplicate_safe"
PREV_FOLDS = PREV / "folds"

OUT = ROOT / "outputs/GUIDE_EXACT_METRICMAX/04_bridge128to8_v1_3_fullgrid"
FINAL = OUT / "final_fit"
WINFOLDS = OUT / "winning_folds"
for p in (OUT, FINAL, WINFOLDS):
    p.mkdir(parents=True, exist_ok=True)

BANK = S3 / "guide_stage03_full_feature_bank.npz"
INDEX_CSV = S3 / "guide_stage03_target_index.csv"

if not BANK.is_file() or not INDEX_CSV.is_file():
    raise FileNotFoundError("Stage-03 guide bank is missing.")
if not PREV_FOLDS.is_dir():
    raise FileNotFoundError("Duplicate-safe Stage04 fold artifacts are missing.")

z = np.load(BANK, allow_pickle=False)
X_ALL = np.asarray(z["X"], np.float32)
UIDS = np.asarray(z["uids"]).astype(str)
FEATURE_NAMES = np.asarray(z["feature_names"]).astype(str)
INDEX = pd.read_csv(INDEX_CSV)

LEARN_MASK = INDEX["record_name"].astype(str).str[0].isin(["a","b","c"]).to_numpy()
TEST_MASK = INDEX["record_name"].astype(str).str.startswith("x").to_numpy()

X_LEARN = X_ALL[LEARN_MASK]
Y_LEARN = INDEX.loc[LEARN_MASK, "y"].to_numpy(np.int8)
R_LEARN = INDEX.loc[LEARN_MASK, "record_name"].astype(str).to_numpy()
UID_LEARN = INDEX.loc[LEARN_MASK, "uid"].astype(str).to_numpy()

X_TEST = X_ALL[TEST_MASK]
UID_TEST = INDEX.loc[TEST_MASK, "uid"].astype(str).to_numpy()

print("Device:", DEVICE)
print("Learn:", X_LEARN.shape, "Test:", X_TEST.shape)
print("Reusing fold artifacts from:", PREV_FOLDS)


In [ ]:

# Cell 2 — load duplicate-safe fold-specific 128-D representations

CONFIGS = {
    "S160":{"anova_k":320,"mrmr_k":224,"shap_k":160},
    "S176":{"anova_k":360,"mrmr_k":240,"shap_k":176},
    "S192":{"anova_k":419,"mrmr_k":272,"shap_k":192},
}
DROPOUTS = [0.3, 0.4, 0.5]

FOLD_BANK = {}
REFERENCE_SPLITS = {}

for cname in CONFIGS:
    for fold in range(5):
        p = PREV_FOLDS / f"{cname}_fold{fold}_drop040.npz"
        if not p.is_file():
            raise FileNotFoundError(p)
        q = np.load(p, allow_pickle=False)
        tr = np.asarray(q["train_rows"], np.int64)
        va = np.asarray(q["val_rows"], np.int64)
        Ztr = np.asarray(q["Ztr128"], np.float32)
        Zv = np.asarray(q["Zv128"], np.float32)
        if Ztr.shape != (len(tr),128) or Zv.shape != (len(va),128):
            raise RuntimeError(f"{cname} fold {fold}: Z128 shape mismatch.")
        FOLD_BANK[(cname,fold)] = (tr,va,Ztr,Zv)

        if fold not in REFERENCE_SPLITS:
            REFERENCE_SPLITS[fold] = (tr.copy(),va.copy())
        else:
            rtr,rva = REFERENCE_SPLITS[fold]
            if not np.array_equal(tr,rtr) or not np.array_equal(va,rva):
                raise RuntimeError(f"Fold {fold}: selector configs do not share the same split.")

# Confirm c05/c06 never cross the split.
for fold,(tr,va) in REFERENCE_SPLITS.items():
    trr=set(R_LEARN[tr]); var=set(R_LEARN[va])
    separated=(("c05" in trr and "c06" in var) or ("c06" in trr and "c05" in var))
    if separated:
        raise RuntimeError(f"Fold {fold}: c05/c06 overlap leakage detected.")

print("All 15 selector-fold Z128 artifacts loaded.")
print("Duplicate-source c05/c06 separation check: PASS")


In [ ]:

# Cell 3 — exact guide bridge and training recipe

class Bridge128to8(nn.Module):
    def __init__(self, dropout=0.4):
        super().__init__()
        self.fc1 = nn.Linear(128,64)
        self.bn1 = nn.BatchNorm1d(64)
        self.fc2 = nn.Linear(64,32)
        self.bn2 = nn.BatchNorm1d(32)
        self.fc3 = nn.Linear(32,8)
        self.head = nn.Linear(8,2)
        self.drop = nn.Dropout(float(dropout))

    def encode(self,x):
        x=self.drop(F.relu(self.bn1(self.fc1(x))))
        x=self.drop(F.relu(self.bn2(self.fc2(x))))
        return self.fc3(x)

    def forward(self,x):
        z=self.encode(x)
        return self.head(z),z

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def calc_metrics(y,p):
    pred=(p>=0.5).astype(np.int8)
    return {
        "accuracy":float(accuracy_score(y,pred)),
        "balanced_accuracy":float(balanced_accuracy_score(y,pred)),
        "precision":float(precision_score(y,pred,zero_division=0)),
        "recall":float(recall_score(y,pred,zero_division=0)),
        "f1":float(f1_score(y,pred,zero_division=0)),
        "auroc":float(roc_auc_score(y,p)),
        "auprc":float(average_precision_score(y,p)),
        "mcc":float(matthews_corrcoef(y,pred)),
    }

def train_bridge(Ztr,ytr,Zv,yv,dropout,seed,epochs=60,batch=512,return_train=False):
    set_seed(seed)
    model=Bridge128to8(dropout).to(DEVICE)

    ds=TensorDataset(
        torch.tensor(Ztr,dtype=torch.float32),
        torch.tensor(ytr,dtype=torch.long)
    )
    gen=torch.Generator().manual_seed(seed)
    dl=DataLoader(ds,batch_size=batch,shuffle=True,generator=gen,num_workers=0)

    opt=torch.optim.AdamW(
        model.parameters(),lr=1e-4,weight_decay=1e-4,
        betas=(0.9,0.999),amsgrad=True
    )

    warmup=10
    def lr_lambda(ep):
        if ep < warmup:
            return (ep+1)/warmup
        progress=(ep-warmup)/max(epochs-warmup-1,1)
        min_ratio=1e-6/1e-4
        return min_ratio+(1-min_ratio)*0.5*(1+math.cos(math.pi*progress))
    sched=torch.optim.lr_scheduler.LambdaLR(opt,lr_lambda=lr_lambda)

    for ep in range(epochs):
        model.train()
        for xb,yb in dl:
            xb=xb.to(DEVICE); yb=yb.to(DEVICE)
            logits,_=model(xb)
            loss=F.cross_entropy(logits,yb,label_smoothing=0.1)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        sched.step()

    @torch.no_grad()
    def infer(X):
        model.eval()
        probs=[]; embs=[]
        for st in range(0,len(X),2048):
            xb=torch.tensor(X[st:st+2048],dtype=torch.float32,device=DEVICE)
            logits,z=model(xb)
            probs.append(torch.softmax(logits,dim=1)[:,1].cpu().numpy())
            embs.append(z.cpu().numpy())
        return np.concatenate(probs),np.concatenate(embs).astype(np.float32)

    pv,Zv8=infer(Zv)
    m=None if yv is None else calc_metrics(yv,pv)

    if return_train:
        ptr,Ztr8=infer(Ztr)
        return model,m,pv,Zv8,ptr,Ztr8
    return model,m,pv,Zv8

print(Bridge128to8())



## Full 3 × 3 guide-permitted combination screen

Every selector/dropout combination is retrained with the same seed for a given fold:

`seed = 1042 + fold`

This means differences across selector/dropout choices are attributable to the choices being compared,
not to deliberately changing the random initialization between candidates.


In [ ]:

# Cell 4 — 9 combinations × 5 folds
GRID_ROWS=[]
BASE_TRAIN_SEED=1042

for cname in CONFIGS:
    for drop in DROPOUTS:
        print(f"\n=== {cname} / dropout={drop:.1f} ===")
        for fold in range(5):
            tr,va,Ztr,Zv=FOLD_BANK[(cname,fold)]
            model,m,pv,Zv8=train_bridge(
                Ztr,Y_LEARN[tr],Zv,Y_LEARN[va],
                dropout=drop,seed=BASE_TRAIN_SEED+fold
            )
            row={"selector":cname,"dropout":drop,"fold":fold,**m}
            GRID_ROWS.append(row)
            print(
                f"fold {fold}: acc={m['accuracy']:.4f} "
                f"auc={m['auroc']:.4f} f1={m['f1']:.4f}"
            )
            del model,pv,Zv8
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

grid=pd.DataFrame(GRID_ROWS)
grid.to_csv(OUT/"fullgrid_fold_metrics.csv",index=False)

summary=grid.groupby(["selector","dropout"],as_index=False).agg(
    mean_accuracy=("accuracy","mean"),
    mean_balanced_accuracy=("balanced_accuracy","mean"),
    mean_precision=("precision","mean"),
    mean_recall=("recall","mean"),
    mean_f1=("f1","mean"),
    mean_auroc=("auroc","mean"),
    mean_auprc=("auprc","mean"),
    mean_mcc=("mcc","mean"),
).sort_values(
    ["mean_accuracy","mean_auroc","mean_f1"],
    ascending=False
).reset_index(drop=True)

summary.to_csv(OUT/"fullgrid_summary.csv",index=False)

print("\nFULL GUIDE-PERMITTED SELECTOR × DROPOUT GRID")
display(summary)

WIN_SELECTOR=str(summary.iloc[0]["selector"])
WIN_DROPOUT=float(summary.iloc[0]["dropout"])
WIN_CFG=CONFIGS[WIN_SELECTOR]

print("\nWINNER:",WIN_SELECTOR,"dropout",WIN_DROPOUT)


In [ ]:

# Cell 5 — regenerate the winning five-fold OOF 8-D representation
oof_prob=np.full(len(Y_LEARN),np.nan,dtype=np.float64)
oof8=np.full((len(Y_LEARN),8),np.nan,dtype=np.float32)

for fold in range(5):
    tr,va,Ztr,Zv=FOLD_BANK[(WIN_SELECTOR,fold)]
    model,m,pv,Zv8,ptr,Ztr8=train_bridge(
        Ztr,Y_LEARN[tr],Zv,Y_LEARN[va],
        dropout=WIN_DROPOUT,seed=BASE_TRAIN_SEED+fold,
        return_train=True
    )

    oof_prob[va]=pv
    oof8[va]=Zv8

    torch.save(
        {
            "state_dict":model.state_dict(),
            "selector":WIN_SELECTOR,
            "dropout":WIN_DROPOUT,
            "fold":fold,
            "seed":BASE_TRAIN_SEED+fold,
        },
        WINFOLDS/f"bridge_fold{fold}.pt"
    )

    np.savez_compressed(
        WINFOLDS/f"bridge_fold{fold}_data.npz",
        train_rows=tr,
        val_rows=va,
        y_train=Y_LEARN[tr],
        y_val=Y_LEARN[va],
        Z128_train=Ztr,
        Z128_val=Zv,
        Z8_train=Ztr8,
        Z8_val=Zv8,
        val_probability=pv,
    )

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if not np.isfinite(oof_prob).all() or not np.isfinite(oof8).all():
    raise RuntimeError("OOF output incomplete.")

OOF_METRICS=calc_metrics(Y_LEARN,oof_prob)
print(json.dumps(OOF_METRICS,indent=2))

pd.DataFrame({
    "uid":UID_LEARN,
    "record_name":R_LEARN,
    "y":Y_LEARN,
    "probability":oof_prob,
}).to_csv(OUT/"winning_bridge_oof_predictions.csv",index=False)

np.savez_compressed(
    OUT/"winning_bridge_oof_8d.npz",
    uids=UID_LEARN.astype("U128"),
    y=Y_LEARN,
    Z8=oof8,
    probability=oof_prob,
)



## Final Stage-03 selector refit for the winning combination

The 9×5 comparison reused the already-computed fold-specific Stage-03 representations. Now the winning
Stage-03 configuration is fitted once on all 35 learning records to produce the final 128-D train/test
representation, followed by the exact 128→64→32→8 bridge.

No official test label is accessed.


In [ ]:

# Cell 6 — exact Stage-03 selector helpers reused from v1.2

def safe_f_scores(X,y):
    f,_=f_classif(X,y)
    return np.nan_to_num(f,nan=0.0,posinf=0.0,neginf=0.0)

def mrmr_greedy(X,y,feature_ids,k,seed):
    X=np.asarray(X,np.float64)
    rel=mutual_info_classif(X,y,random_state=seed)
    rel=np.nan_to_num(rel,nan=0.0)
    if rel.max()>rel.min():
        rel=(rel-rel.min())/(rel.max()-rel.min())
    else:
        rel=np.zeros_like(rel)

    C=np.corrcoef(X,rowvar=False)
    C=np.nan_to_num(np.abs(C),nan=0.0,posinf=0.0,neginf=0.0)
    np.fill_diagonal(C,0.0)

    selected=[]
    remaining=np.ones(X.shape[1],dtype=bool)
    redsum=np.zeros(X.shape[1],dtype=np.float64)

    first=int(np.argmax(rel))
    selected.append(first); remaining[first]=False
    redsum+=C[:,first]

    while len(selected)<min(k,X.shape[1]):
        score=rel-redsum/max(len(selected),1)
        score[~remaining]=-np.inf
        j=int(np.argmax(score))
        selected.append(j); remaining[j]=False
        redsum+=C[:,j]

    selected=np.asarray(selected,np.int64)
    return np.asarray(feature_ids,np.int64)[selected]

def shap_rank(X,y,feature_ids,seed):
    model=XGBClassifier(
        n_estimators=320,max_depth=5,learning_rate=0.04,
        subsample=0.90,colsample_bytree=0.90,min_child_weight=3,
        reg_lambda=2.0,reg_alpha=0.05,eval_metric="logloss",
        tree_method="hist",random_state=seed,n_jobs=-1
    )
    model.fit(X,y)
    contrib=model.get_booster().predict(xgb.DMatrix(X),pred_contribs=True)
    imp=np.mean(np.abs(contrib[:,:-1]),axis=0)
    order=np.argsort(-imp)
    return np.asarray(feature_ids,np.int64)[order]

class GuideSelector128:
    def __init__(self,cfg,seed=42):
        self.cfg=dict(cfg); self.seed=int(seed)

    def fit(self,X,y):
        self.imputer=SimpleImputer(strategy="median",add_indicator=False)
        Xi=self.imputer.fit_transform(X)

        f=safe_f_scores(Xi,y)
        ak=min(int(self.cfg["anova_k"]),Xi.shape[1])
        self.anova_ids=np.argsort(-f)[:ak].astype(np.int64)

        self.pre_scale=StandardScaler()
        Xa=self.pre_scale.fit_transform(Xi[:,self.anova_ids])

        self.mrmr_ids=mrmr_greedy(
            Xa,y,self.anova_ids,
            min(int(self.cfg["mrmr_k"]),len(self.anova_ids)),
            self.seed
        )

        pos={int(fid):i for i,fid in enumerate(self.anova_ids)}
        mpos=np.asarray([pos[int(fid)] for fid in self.mrmr_ids],np.int64)
        Xm=Xa[:,mpos]

        ranked=shap_rank(Xm,y,self.mrmr_ids,self.seed)
        self.shap_ids=ranked[:min(int(self.cfg["shap_k"]),len(ranked))]

        spos=np.asarray([pos[int(fid)] for fid in self.shap_ids],np.int64)
        Xs=Xa[:,spos]

        self.final_scale=StandardScaler()
        Xss=self.final_scale.fit_transform(Xs)

        self.pca=PCA(n_components=128,whiten=True,random_state=self.seed)
        Z=self.pca.fit_transform(Xss)
        self.explained_128=float(self.pca.explained_variance_ratio_.sum())
        return Z.astype(np.float32)

    def transform(self,X):
        Xi=self.imputer.transform(X)
        Xa=self.pre_scale.transform(Xi[:,self.anova_ids])
        pos={int(fid):i for i,fid in enumerate(self.anova_ids)}
        spos=np.asarray([pos[int(fid)] for fid in self.shap_ids],np.int64)
        Xs=Xa[:,spos]
        Xss=self.final_scale.transform(Xs)
        return self.pca.transform(Xss).astype(np.float32)


In [ ]:

# Cell 7 — final all-learning fit and quantum-ready 8-D artifact

FINAL_SELECTOR=GuideSelector128(WIN_CFG,seed=SEED)
Z128_LEARN=FINAL_SELECTOR.fit(X_LEARN,Y_LEARN)
Z128_TEST=FINAL_SELECTOR.transform(X_TEST)

FINAL_MODEL,_,_,_,_,Z8_LEARN = train_bridge(
    Z128_LEARN,Y_LEARN,Z128_LEARN,None,
    dropout=WIN_DROPOUT,seed=2042,
    return_train=True
)

@torch.no_grad()
def encode_only(model,X):
    model.eval()
    out=[]
    for st in range(0,len(X),2048):
        xb=torch.tensor(X[st:st+2048],dtype=torch.float32,device=DEVICE)
        out.append(model.encode(xb).cpu().numpy())
    return np.concatenate(out).astype(np.float32)

Z8_TEST=encode_only(FINAL_MODEL,Z128_TEST)

torch.save(
    {
        "state_dict":FINAL_MODEL.state_dict(),
        "selector":WIN_SELECTOR,
        "selector_config":WIN_CFG,
        "dropout":WIN_DROPOUT,
        "seed":2042,
    },
    FINAL/"bridge128to8_model.pt"
)

np.savez_compressed(
    FINAL/"quantum_ready_8d.npz",
    learn_uids=UID_LEARN.astype("U128"),
    test_uids=UID_TEST.astype("U128"),
    y_learn=Y_LEARN,
    Z128_learn=Z128_LEARN,
    Z128_test=Z128_TEST,
    Z8_learn=Z8_LEARN,
    Z8_test=Z8_TEST,
)

pd.DataFrame({
    "feature_id":FINAL_SELECTOR.shap_ids,
    "feature_name":FEATURE_NAMES[FINAL_SELECTOR.shap_ids],
}).to_csv(FINAL/"shap_selected_features.csv",index=False)

print("Final selector:",WIN_SELECTOR,WIN_CFG)
print("Final dropout:",WIN_DROPOUT)
print("PCA128 explained variance:",FINAL_SELECTOR.explained_128)
print("Learn 8-D:",Z8_LEARN.shape)
print("Official test 8-D:",Z8_TEST.shape)
print("Official test metric computed: NO")


In [ ]:

# Cell 8 — manifest
def sha256_file(p,chunk=1<<20):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

qpath=FINAL/"quantum_ready_8d.npz"
mpath=FINAL/"bridge128to8_model.pt"

manifest={
    "pipeline":"QML-SleepNet guide Stage04 bridge full-grid metric-max v1",
    "selector_candidates":CONFIGS,
    "dropout_candidates":DROPOUTS,
    "comparison":"complete 3x3 selector x dropout grid",
    "selection_order":["mean_accuracy","mean_auroc","mean_f1"],
    "same_fold_seed_across_candidates":True,
    "winning_selector":WIN_SELECTOR,
    "winning_selector_config":WIN_CFG,
    "winning_dropout":WIN_DROPOUT,
    "fivefold_oof_metrics":OOF_METRICS,
    "c05_c06_duplicate_safe_folds_reused":True,
    "official_test_labels_used":False,
    "quantum_ready_8d_path":str(qpath),
    "quantum_ready_8d_sha256":sha256_file(qpath),
    "bridge_model_path":str(mpath),
    "bridge_model_sha256":sha256_file(mpath),
    "next":"Guide Stage04 quantum module: Angle/IQP, VQC, quantum kernel/QSVC, quantum transformer",
}
(OUT/"STAGE04_FULLGRID_MANIFEST.json").write_text(json.dumps(manifest,indent=2))

print("="*110)
print("GUIDE STAGE04 BRIDGE FULL-GRID METRICMAX COMPLETE")
print("="*110)
print("Winner:",WIN_SELECTOR,"dropout",WIN_DROPOUT)
print("OOF metrics:",OOF_METRICS)
print("Quantum-ready artifact:",qpath)
print("NEXT: exact guide QML block.")
